# Verification Notebook
This notebook contains the verification tests performed for our MILP problem.

They follow from Sargeant 1998 criteria for verification and validation of simulation models.

Three types of tests are proposed:
1. Unit tests on the deterministic pre-processing.
2. Extreme/Degenerate condition tests.
3. Infeasibility tests.

In [1]:
import numpy as np

## 1. Unit Tests (Deterministic Pre-Processing)

### phi() - Rocket Equation

In [2]:
# Dummy Network
Connections = {0: [0,1], 1: [0,1]}      # 2 nodes
g_0 = 9.80665                           # m/s^2

delta_V = {0: {0: 0, 1: 2.0},                # ΔV [km/s]
           1: {0: 2.0, 1: 0}}

# Dummy Vehicle Set
I_sp    = np.array([0, 300])

In [3]:
# Phi Function Definition - As in MILP code
def phi(i, j, v, dV=delta_V, I_sp=I_sp, g_0=g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000 * dV[i][j] / (I_sp[v] * g_0)))

In [4]:
# TEST 1: Zero-Isp Special Case

test_1 = phi(0, 1, 0);              # node 0 to node 1; vehicle 0 (Isp = 0s)
expected_1 = 1

assert test_1 == expected_1, f"TEST 1 - FAIL: expected {expected_1}, got {test_1}"
print(f"TEST 1 - PASS: phi(Isp=0) = {test_1} (expected {expected_1})")

TEST 1 - PASS: phi(Isp=0) = 1 (expected 1)


In [5]:
# TEST 2: Manual Rederivation of the Rocket Equation

i, j, v = 0, 1, 1                   # node 0 to node 1; vehicle 1 (Isp = 300s)
dV_ms = delta_V[i][j] * 1000        # km/s to m/s conversion
expected_2 = 1 - np.exp(-dV_ms / (I_sp[v] * g_0))

test_2 = phi(i, j, v)

assert test_2 == expected_2, f"TEST 2 - FAIL: expected {expected_2}, got {test_2}"
print(f"TEST 2 - PASS: phi = {test_2:.4f} (expected {expected_2:.4f})")

TEST 2 - PASS: phi = 0.4933 (expected 0.4933)


In [6]:
# TEST 3: Holdover Arc
# Staying at the same node should never burn propellant, for any vehicle

result_v0 = phi(0, 0, 0)            # Node 0 to Node 0; Vehicle 0 (Isp = 0s)
result_v1 = phi(0, 0, 1)            # Node 0 to Node 0, Vehicle 1 (Isp = 300s)

assert result_v0 == 1, f"TEST 3 - FAIL: expected 1 for vehicle 0 (Isp = 0s), got {result_v0}"
assert result_v1 == 0, f"TEST 3 - FAIL: expected 0 for vehicle 1 (Isp = 300s) and zero dV, got {result_v1}"

print(f"TEST 3 - PASS: phi(holdover, Isp=0s) = {result_v0}, phi(holdover, Isp=300s) = {result_v1}")

TEST 3 - PASS: phi(holdover, Isp=0s) = 1, phi(holdover, Isp=300s) = 0.0


### AllPossibleOutflowArcs()

In [7]:
# AllpossibleOutflowArcs function from MILP code

def AllpossibleOutflowArcs(Connections, T_adv, window, TOFused):
  

  AllArcs = {}

  for t in T_adv:
    TimeNode = {}
    for i in Connections:
      if t in window[i]:
        
        Now = window[i].index(t)
        TimeNode[i] = {}
        
        for j in Connections[i]:
            
            if t+TOFused[i][j] in window[j]:
              TimeNode[i][j]= {"ArrivalTime": t+TOFused[i][j], "FullTravelTime":TOFused[i][j] }

        #If the holding arc is not available, then the arc to the next available
        #free time block is added, as long as we are not at the end of the window (N_Window[i])
        if (i not in TimeNode[i]) and (Now+1 != len(window[i])): 
          TimeNode[i][i]={"ArrivalTime":window[i][Now+1],"FullTravelTime":window[i][Now+1] - t}
        
        
    if TimeNode != {}:
      AllArcs[t] = TimeNode
            
                    
    
  return AllArcs

In [8]:
# TEST 4: Full Hand-Traced Network

Connection_4    = {0: [0, 1], 1: [0, 1]}
N_Window_4      = {0: [0, 2], 1: [0, 3]}
TOF_4           = {0: {0: 1, 1: 2}, 1: {0: 2, 1: 1}}
T_adv_4         = [0, 1, 2, 3]

result_4 = AllpossibleOutflowArcs(Connection_4, T_adv_4, window=N_Window_4, TOFused=TOF_4)

expected_4 = {
    0: {
        0: {0: {"ArrivalTime": 2, "FullTravelTime": 2}},
        1: {0: {"ArrivalTime": 2, "FullTravelTime": 2},
            1: {"ArrivalTime": 3, "FullTravelTime": 3}},
    },
    2: {0: {}},
    3: {1: {}},
}

assert result_4 == expected_4, f"TEST 4 - FAIL: \ngot {result_4}\nexpected {expected_4}"
print("TEST 4 - PASS: full hand-traced arc set matches exactly")
print(result_4)

TEST 4 - PASS: full hand-traced arc set matches exactly
{0: {0: {0: {'ArrivalTime': 2, 'FullTravelTime': 2}}, 1: {0: {'ArrivalTime': 2, 'FullTravelTime': 2}, 1: {'ArrivalTime': 3, 'FullTravelTime': 3}}}, 2: {0: {}}, 3: {1: {}}}


In [9]:
# TEST 5: Arrival Outside Destination Window

Connections_5   = {0: [1], 1: [0]}
N_Window_5      = {0: [0], 1: [5]}
TOF_5           = {0: {1: 2}, 1: {0: 2}}
T_adv_5         = list(range(6))

result_5 = AllpossibleOutflowArcs(Connections_5, T_adv_5, window=N_Window_5, TOFused=TOF_5)

assert 0 in result_5, "TEST 5 - FAIL: expected an entry for t=0"
assert result_5[0][0] == {}, f"TEST 5 -FAIL: expected no reachable arcs from node 0 at t=0, got {result_5[0][0]}"
assert 1 not in result_5[0][0], "TEST 5 -FAIL: arc to node 1 should not exist -- arrival time misses its window"

print("TEST 5 - PASS: arc correctly absent when arrival time falls outside destination window.")
print(result_5)

TEST 5 - PASS: arc correctly absent when arrival time falls outside destination window.
{0: {0: {}}, 5: {1: {}}}


In [10]:
# TEST 6: Holdover Fallback, Unequal Gap Sizes

Connections_6   = {0: [0]}
N_Window_6      = {0: [0, 4, 9]}
TOF_6           = {0: {0: 100}}
T_adv_6         = list(range(10))

result_6 = AllpossibleOutflowArcs(Connections_6, T_adv_6, window=N_Window_6, TOFused=TOF_6)

expected_6 = {
    0: {0: {0: {"ArrivalTime": 4, "FullTravelTime": 4}}},
    4: {0: {0: {"ArrivalTime": 9, "FullTravelTime": 5}}},
    9: {0: {}},
}

assert result_6 == expected_6, f"TEST 6 - FAIL:\ngot {result_6}\nexpected {expected_6}"

print(f"TEST 6 - PASS: Holdover fallback arcs correctly sized for unequal window gaps, and correctly withheld at the final window entry")
print(result_6)

TEST 6 - PASS: Holdover fallback arcs correctly sized for unequal window gaps, and correctly withheld at the final window entry
{0: {0: {0: {'ArrivalTime': 4, 'FullTravelTime': 4}}}, 4: {0: {0: {'ArrivalTime': 9, 'FullTravelTime': 5}}}, 9: {0: {}}}


### Solo_SC_Consumption_NodV()

In [20]:
# Dummy Commodity Set
crew_mass       = 100
consumption     = 2.0       # kg/crew/day
PropIndex       = 4
CommodityMassConversion = [crew_mass, 1, 1, 1, 1]
Carriable = {0: "INTEGER", 1: "INTEGER"}

# Addition Dummy Network
TOF = {0: {0: 1, 1: 1},
       1: {0: 1, 1: 1}}

# Addition Dummy Vehicle Set
StructureMass = np.array([1000, 2000])

In [19]:
# Solo_SC_Consumption_NodV function from MILP code

def Solo_SC_Consumption_NodV(i, j, consumption=consumption, TOF=TOF, extraPayload=Carriable):

    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock

    for i1,(x1) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1


    return FullMatrix

In [13]:
# TEST 7: Matrix Structure, Normal Usage

i, j = 0, 0
matrix_nodv = Solo_SC_Consumption_NodV(i, j)

expected_nodv = np.zeros((8, 8))
CommodityBlock_expected = np.array([
    [1, 0, 0, 0, 0, 0],                                 # crew
    [-consumption * TOF[i][j], 1, 0, 0, 0, 0],          # consumables
    [0, 0, 1, 0, 0, 0],                                 # equipment
    [0, 0, 0, 1, 0, 0],                                 # samples
    [0, 0, 0, 0, 1, 0],                                 # propellant (unchanged -- no burn) 
    [0, 0, 0, 0, 0, 1],                                 # spacecraft count
])
expected_nodv[:6, :6]   = CommodityBlock_expected
expected_nodv[6,6]      = 1
expected_nodv[7,7]      = 1

assert np.allclose(matrix_nodv, expected_nodv), f"TEST 7 - FAIL:\ngot\n{matrix_nodv}\nexpected:\n{expected_nodv}"
print("TEST 7 - PASS: Holdover consumption matrix matches Eq.(8) with phi=0 (no burn)")

TEST 7 - PASS: Holdover consumption matrix matches Eq.(8) with phi=0 (no burn)


### Solo_SC_Consumption()

In [21]:
# Solo_SC_Consumption function from MILP code

def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass, extraPayload = Carriable,PropellantIndex = PropIndex):


    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft

    
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock




    
    for i1,(x1,vtype) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1
        FullMatrix[PropellantIndex, -i2] = -1*structure_mass[x1]*phi(i,j,v,delta_V,I_sp,g_0)


    return FullMatrix


In [ ]:
# TEST 8: Propellant Row Matches Hand-Derived Rocket Equation

i, j, v = 0, 1, 1
dV_ms = delta_V[i][j] * 1000
phi_expected = 1 - np.exp(-dV_ms / (I_sp[v] * g_0))

matrix_burn = Solo_SC_Consumption(i, j, v)

expected_row = np.array([
    -crew_mass * phi_expected,
    -phi_expected,
    -phi_expected,
    -phi_expected,
    1 - phi_expected,
    -StructureMass[v] * phi_expected,
    -StructureMass[0] * phi_expected,
    -StructureMass[1] * phi_expected,
])

assert np.allclose(matrix_burn[4,:], expected_row), f"TEST 8 - FAIL:\ngot: {matrix_burn[4,:]}\nexpected: {expected_row}"
print(f"TEST 8 - PASS: Propellant row matches hand-derived rocket equation (phi={phi_expected:.4f}).")

TEST 8 - PASS: Propellant row matches hand-derived rocket equation (phi=0.4933).


In [25]:
# TEST 9: Isp=0 special case

i, j, v = 0, 1, 0
matrix_isp0 = Solo_SC_Consumption(i, j, v)

assert matrix_isp0[4,4] == 0, f"TEST 9 - FAIL: Propellant self-retention term = {matrix_isp0[4,4]}, expected 0."
assert matrix_isp0[4,5] == -StructureMass[v], f"TEST 9 - FAIL: Own structure mass term = {matrix_isp0[4,5]}, expected {-StructureMass[v]}."

print("TEST 9 - PASS: Isp=0 vehicle forces phi=1. Propellant row's self-retention term is exactly 0.")

TEST 9 - PASS: Isp=0 vehicle forces phi=1. Propellant row's self-retention term is exactly 0.
